# 04 - Embeddings e indexación vectorial

Este notebook genera embeddings a partir de los chunks estructurados del corpus normativo ambiental e indexa los resultados en una base vectorial usando **Qdrant**.

Entrada principal:

- `data/chunks/chunks_normativa_v1.jsonl`

Salida generada:

- colección vectorial en Qdrant;
- reporte de indexación en `experiments/resultados/`;
- prueba inicial de búsqueda semántica top-k.

Este paso permite pasar de documentos segmentados a una base consultable por similitud semántica.


## 0. Requisito previo: levantar Qdrant

Antes de ejecutar la indexación, asegúrate de tener Qdrant corriendo.

Con Docker:

```bash
docker run -p 6333:6333 -p 6334:6334 qdrant/qdrant
```

Luego abre en el navegador:

```text
http://localhost:6333/dashboard
```

Si no usas Docker todavía, primero instala Docker Desktop o consulta con el equipo para definir dónde correrá Qdrant.


## 1. Importación de librerías

In [5]:
# ============================================================
# 0. Verificación e instalación de librerías necesarias
# ============================================================

import sys
import subprocess
import importlib.util

def instalar_paquete(nombre_import, comando_pip):
    """
    Verifica si una librería está instalada.
    Si no está, intenta instalarla y muestra el error real si falla.
    """
    if importlib.util.find_spec(nombre_import) is not None:
        print(f"{nombre_import} ya está instalado.")
        return True

    print(f"Falta {nombre_import}. Instalando...")

    resultado = subprocess.run(
        [sys.executable, "-m", "pip"] + comando_pip,
        capture_output=True,
        text=True
    )

    if resultado.returncode == 0:
        print(f"{nombre_import} instalado correctamente.")
        return True
    else:
        print("=" * 80)
        print(f"ERROR instalando {nombre_import}")
        print("=" * 80)
        print("STDOUT:")
        print(resultado.stdout)
        print("STDERR:")
        print(resultado.stderr)
        print("=" * 80)
        return False


# Actualizar herramientas base de instalación
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    text=True
)

# Librerías normales
instalar_paquete("pandas", ["install", "pandas"])
instalar_paquete("numpy", ["install", "numpy"])
instalar_paquete("tqdm", ["install", "tqdm"])

# PyTorch CPU para Windows / VS Code / notebook local
instalar_paquete(
    "torch",
    [
        "install",
        "torch",
        "torchvision",
        "torchaudio",
        "--index-url",
        "https://download.pytorch.org/whl/cpu"
    ]
)

# Librerías del notebook 04
instalar_paquete("sentence_transformers", ["install", "sentence-transformers"])
instalar_paquete("qdrant_client", ["install", "qdrant-client"])

print("Verificación finalizada.")

pandas ya está instalado.
numpy ya está instalado.
tqdm ya está instalado.
Falta torch. Instalando...
torch instalado correctamente.
Falta sentence_transformers. Instalando...
sentence_transformers instalado correctamente.
Falta qdrant_client. Instalando...
qdrant_client instalado correctamente.
Verificación finalizada.


In [1]:
# ============================================================
# 1. Importación de librerías
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import time
import uuid

try:
    import pandas as pd
    print("pandas importado correctamente.")
except ImportError:
    raise ImportError("No se encontró pandas. Instálalo con: pip install pandas")

try:
    import numpy as np
    print("numpy importado correctamente.")
except ImportError:
    raise ImportError("No se encontró numpy. Instálalo con: pip install numpy")

try:
    from tqdm.auto import tqdm
    print("tqdm importado correctamente.")
except ImportError:
    raise ImportError("No se encontró tqdm. Instálalo con: pip install tqdm")

try:
    from sentence_transformers import SentenceTransformer
    print("sentence-transformers importado correctamente.")
except ImportError:
    raise ImportError(
        "No se encontró sentence-transformers. Instálalo con: pip install sentence-transformers"
    )

try:
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams, PointStruct
    print("qdrant-client importado correctamente.")
except ImportError:
    raise ImportError(
        "No se encontró qdrant-client. Instálalo con: pip install qdrant-client"
    )

print("Librerías importadas correctamente.")


pandas importado correctamente.
numpy importado correctamente.
tqdm importado correctamente.


c:\Users\lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers importado correctamente.
qdrant-client importado correctamente.
Librerías importadas correctamente.


## 2. Definición de rutas del proyecto

El notebook puede ejecutarse desde la raíz del repositorio o desde la carpeta `notebooks/`.


In [3]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    ROOT_DIR = current_dir.parent
else:
    ROOT_DIR = current_dir

DATA_DIR = ROOT_DIR / "data"
CHUNKS_DIR = DATA_DIR / "chunks"
REPORTS_DIR = ROOT_DIR / "experiments" / "resultados"

CHUNKS_JSONL_PATH = CHUNKS_DIR / "chunks_normativa_v1.jsonl"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"CHUNKS_JSONL_PATH: {CHUNKS_JSONL_PATH}")
print(f"REPORTS_DIR: {REPORTS_DIR}")


ROOT_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana
CHUNKS_JSONL_PATH: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\data\chunks\chunks_normativa_v1.jsonl
REPORTS_DIR: c:\Users\lenovo\Downloads\2026\2026 - A\TABD\Repo_TABD\rag-normativa-ambiental-peruana\experiments\resultados


## 3. Configuración general

Se define el modelo de embeddings, el nombre de la colección en Qdrant y los parámetros de indexación.

Modelo principal definido para el proyecto:

```text
BAAI/bge-m3
```

Si tu computadora demora demasiado o no tiene recursos suficientes, puedes cambiar temporalmente a:

```text
intfloat/multilingual-e5-base
```


In [4]:
# ============================================================
# 3. Configuración general
# ============================================================

MODEL_NAME = "BAAI/bge-m3"
# MODEL_NAME = "intfloat/multilingual-e5-base"  # alternativa más ligera

COLLECTION_NAME = "normativa_ambiental_chunks_v1"

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

BATCH_SIZE = 16
TOP_K = 5

# Para BGE-M3 se puede dejar sin prefijos.
# Si usas multilingual-e5-base, puedes usar:
# PASSAGE_PREFIX = "passage: "
# QUERY_PREFIX = "query: "
PASSAGE_PREFIX = ""
QUERY_PREFIX = ""

print(f"Modelo de embeddings: {MODEL_NAME}")
print(f"Colección Qdrant: {COLLECTION_NAME}")
print(f"Qdrant: {QDRANT_HOST}:{QDRANT_PORT}")


Modelo de embeddings: BAAI/bge-m3
Colección Qdrant: normativa_ambiental_chunks_v1
Qdrant: localhost:6333


## 4. Carga de chunks generados

In [5]:
if not CHUNKS_JSONL_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo de chunks: {CHUNKS_JSONL_PATH}. "
        "Ejecuta primero el Notebook 03 de chunking estructural."
    )

chunks = []

with open(CHUNKS_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print(f"Total de chunks cargados: {len(chunks)}")

chunks_df = pd.DataFrame(chunks)
chunks_df.head()


Total de chunks cargados: 1057


,id_chunk,id_documento,archivo_pdf,archivo_txt,titulo_documento,tipo_norma,numero_norma,entidad_emisora,fecha_publicacion,tema_principal,...,titulo_seccion,capitulo,anexo,pagina_inicio,pagina_fin,orden_chunk,suborden,n_palabras,n_caracteres,texto
0,DOC-001_CHK-0001,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,1,1,453,2860,"NORMAS LEGALES El Peruano Lima, jueves 6 de se..."
1,DOC-001_CHK-0001_P02,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,1,2,295,1878,así como en el proceso al que se reﬁ ere el De...
2,DOC-001_CHK-0002,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,2,1,58,402,Artículo 1º.- Del objeto de la norma\nApruébes...
3,DOC-001_CHK-0003,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,3,1,25,144,Artículo 2º.- De la vigencia\nEl presente Decr...
4,DOC-001_CHK-0004,DOC-001,DOC-001_DS_004_2012_MINAM_IGAC.pdf,DOC-001.txt,Aprueban Disposiciones Complementarias para el...,Decreto Supremo,Decreto Supremo N.º 004-2012-MINAM,MINAM,2012-09-06,Gestión ambiental,...,No determinado,No determinado,No determinado,1,1,4,1,51,311,Artículo 3º.- Del refrendo\nEl presente Decret...


## 5. Validación básica de chunks

Se revisa que los chunks tengan texto y metadatos mínimos antes de generar embeddings.


In [6]:
required_chunk_fields = [
    "id_chunk",
    "id_documento",
    "titulo_documento",
    "tipo_norma",
    "numero_norma",
    "tema_principal",
    "subtema",
    "texto",
]

missing_fields = [field for field in required_chunk_fields if field not in chunks_df.columns]

if missing_fields:
    raise ValueError(f"Faltan campos obligatorios en los chunks: {missing_fields}")

empty_text_chunks = chunks_df[chunks_df["texto"].isna() | (chunks_df["texto"].astype(str).str.strip() == "")]

if not empty_text_chunks.empty:
    print(f"Advertencia: existen {len(empty_text_chunks)} chunks sin texto. Serán excluidos.")
    chunks_df = chunks_df[~chunks_df.index.isin(empty_text_chunks.index)].copy()

chunks_df = chunks_df.reset_index(drop=True)

print(f"Chunks válidos para embeddings: {len(chunks_df)}")
print("Distribución de palabras por chunk:")
display(chunks_df["n_palabras"].describe())


Chunks válidos para embeddings: 1057
Distribución de palabras por chunk:


count    1057.000000
mean      210.133396
std       163.340571
min        25.000000
25%        72.000000
50%       139.000000
75%       400.000000
max       635.000000
Name: n_palabras, dtype: float64

## 6. Carga del modelo de embeddings

La primera vez puede demorar porque el modelo se descarga desde Hugging Face.

Si esta celda demora demasiado, es normal en la primera ejecución. Si falla por memoria o tiempo, cambia temporalmente el modelo a `intfloat/multilingual-e5-base`.


In [ ]:
# ============================================================
# 6. Carga del modelo de embeddings con FlagEmbedding
# ============================================================

import time
import torch
from FlagEmbedding import BGEM3FlagModel

start_time = time.time()

device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = True if device == "cuda" else False

print(f"Dispositivo usado: {device}")
print(f"FP16 activado: {use_fp16}")

model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=use_fp16,
    device=device
)

load_time = time.time() - start_time

print(f"Modelo BGE-M3 cargado correctamente en {load_time:.2f} segundos.")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6549.73it/s]


: 

## 7. Preparación de textos para embeddings

Se combina el texto del chunk con algunos metadatos útiles para reforzar el contexto semántico.


In [ ]:
def build_embedding_text(row) -> str:
    metadata_context = (
        f"Título del documento: {row.get('titulo_documento', 'No determinado')}\n"
        f"Tipo de norma: {row.get('tipo_norma', 'No determinado')}\n"
        f"Número de norma: {row.get('numero_norma', 'No determinado')}\n"
        f"Tema: {row.get('tema_principal', 'No determinado')}\n"
        f"Subtema: {row.get('subtema', 'No determinado')}\n"
        f"Sección: {row.get('seccion', 'No determinado')}\n"
    )
    return PASSAGE_PREFIX + metadata_context + "\nContenido:\n" + str(row.get("texto", ""))

chunks_df["texto_embedding"] = chunks_df.apply(build_embedding_text, axis=1)

print("Ejemplo de texto usado para embedding:")
print(chunks_df.loc[0, "texto_embedding"][:1500])


## 8. Generación de embeddings

Se generan embeddings por lotes para evitar consumir demasiada memoria.


In [ ]:
texts = chunks_df["texto_embedding"].tolist()

embeddings = []

start_time = time.time()

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Generando embeddings"):
    batch_texts = texts[start:start + BATCH_SIZE]
    batch_embeddings = model.encode(
        batch_texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    embeddings.extend(batch_embeddings)

embedding_time = time.time() - start_time

embeddings = np.array(embeddings, dtype=np.float32)

print(f"Embeddings generados: {embeddings.shape}")
print(f"Tiempo total de generación: {embedding_time:.2f} segundos")
print(f"Tiempo promedio por chunk: {embedding_time / max(len(texts), 1):.4f} segundos")


## 9. Conexión con Qdrant

Se verifica que Qdrant esté corriendo en `localhost:6333`.


In [ ]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

try:
    collections = client.get_collections()
    print("Conexión con Qdrant establecida correctamente.")
    print(collections)
except Exception as e:
    raise ConnectionError(
        "No se pudo conectar con Qdrant. Asegúrate de ejecutarlo con Docker:\n"
        "docker run -p 6333:6333 -p 6334:6334 qdrant/qdrant"
    ) from e


## 10. Creación o recreación de la colección

Para evitar duplicados durante pruebas, se recrea la colección.

Si quieres conservar una colección previa, cambia `RECREATE_COLLECTION = False`.


In [ ]:
RECREATE_COLLECTION = True

existing_collections = [collection.name for collection in client.get_collections().collections]

if COLLECTION_NAME in existing_collections and RECREATE_COLLECTION:
    client.delete_collection(collection_name=COLLECTION_NAME)
    print(f"Colección eliminada: {COLLECTION_NAME}")

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE
    )
)

print(f"Colección lista: {COLLECTION_NAME}")


## 11. Preparación de puntos para Qdrant

Cada chunk se almacena como un punto vectorial.  
El vector representa el contenido semántico y el payload conserva los metadatos.


In [ ]:
def clean_payload_value(value):
    if pd.isna(value):
        return "No determinado"
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value

points = []

for idx, row in chunks_df.iterrows():
    payload = {
        "id_chunk": clean_payload_value(row.get("id_chunk")),
        "id_documento": clean_payload_value(row.get("id_documento")),
        "archivo_pdf": clean_payload_value(row.get("archivo_pdf")),
        "titulo_documento": clean_payload_value(row.get("titulo_documento")),
        "tipo_norma": clean_payload_value(row.get("tipo_norma")),
        "numero_norma": clean_payload_value(row.get("numero_norma")),
        "entidad_emisora": clean_payload_value(row.get("entidad_emisora")),
        "fecha_publicacion": clean_payload_value(row.get("fecha_publicacion")),
        "tema_principal": clean_payload_value(row.get("tema_principal")),
        "subtema": clean_payload_value(row.get("subtema")),
        "fuente_oficial": clean_payload_value(row.get("fuente_oficial")),
        "url_fuente": clean_payload_value(row.get("url_fuente")),
        "estado_vigencia": clean_payload_value(row.get("estado_vigencia")),
        "prioridad": clean_payload_value(row.get("prioridad")),
        "tipo_chunk": clean_payload_value(row.get("tipo_chunk")),
        "seccion": clean_payload_value(row.get("seccion")),
        "titulo_seccion": clean_payload_value(row.get("titulo_seccion")),
        "capitulo": clean_payload_value(row.get("capitulo")),
        "anexo": clean_payload_value(row.get("anexo")),
        "pagina_inicio": clean_payload_value(row.get("pagina_inicio")),
        "pagina_fin": clean_payload_value(row.get("pagina_fin")),
        "n_palabras": clean_payload_value(row.get("n_palabras")),
        "n_caracteres": clean_payload_value(row.get("n_caracteres")),
        "texto": clean_payload_value(row.get("texto")),
    }

    point = PointStruct(
        id=idx,
        vector=embeddings[idx].tolist(),
        payload=payload
    )
    points.append(point)

print(f"Puntos preparados para Qdrant: {len(points)}")


## 12. Indexación en Qdrant

Se suben los puntos vectoriales por lotes.


In [ ]:
start_time = time.time()

for start in tqdm(range(0, len(points), BATCH_SIZE), desc="Indexando en Qdrant"):
    batch_points = points[start:start + BATCH_SIZE]
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch_points
    )

indexing_time = time.time() - start_time

print(f"Indexación completada en {indexing_time:.2f} segundos.")
print(f"Total de puntos indexados: {len(points)}")


## 13. Verificación de la colección indexada

In [ ]:
collection_info = client.get_collection(collection_name=COLLECTION_NAME)

print(collection_info)


## 14. Función de búsqueda semántica

Esta función convierte una consulta en embedding y recupera los chunks más similares desde Qdrant.


In [ ]:
def search_semantic(query: str, top_k: int = TOP_K):
    query_text = QUERY_PREFIX + query

    query_vector = model.encode(
        [query_text],
        normalize_embeddings=True,
        show_progress_bar=False
    )[0].astype(np.float32)

    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector.tolist(),
        limit=top_k,
        with_payload=True
    )

    output = []

    for rank, result in enumerate(results, start=1):
        payload = result.payload

        output.append({
            "rank": rank,
            "score": result.score,
            "id_chunk": payload.get("id_chunk"),
            "id_documento": payload.get("id_documento"),
            "titulo_documento": payload.get("titulo_documento"),
            "tipo_norma": payload.get("tipo_norma"),
            "numero_norma": payload.get("numero_norma"),
            "tema_principal": payload.get("tema_principal"),
            "subtema": payload.get("subtema"),
            "seccion": payload.get("seccion"),
            "pagina_inicio": payload.get("pagina_inicio"),
            "pagina_fin": payload.get("pagina_fin"),
            "texto": payload.get("texto"),
        })

    return output


## 15. Prueba inicial de búsqueda vectorial

In [ ]:
test_queries = [
    "¿Qué regula el Instrumento de Gestión Ambiental Correctivo IGAC?",
    "¿Qué normas tratan sobre estándares de calidad ambiental?",
    "¿Qué obligaciones ambientales existen para la pequeña minería?",
]

for query in test_queries:
    print("=" * 120)
    print(f"CONSULTA: {query}")
    print("=" * 120)

    results = search_semantic(query, top_k=5)

    for item in results:
        print(f"Rank: {item['rank']} | Score: {item['score']:.4f}")
        print(f"Documento: {item['id_documento']} - {item['titulo_documento']}")
        print(f"Norma: {item['numero_norma']}")
        print(f"Sección: {item['seccion']}")
        print(f"Páginas: {item['pagina_inicio']} - {item['pagina_fin']}")
        print("-" * 80)
        print(str(item["texto"])[:700])
        print()


## 16. Guardado de reporte de indexación

Se guarda un resumen técnico de la indexación realizada.


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

report = {
    "fecha_indexacion": datetime.now().isoformat(timespec="seconds"),
    "modelo_embeddings": MODEL_NAME,
    "coleccion_qdrant": COLLECTION_NAME,
    "vector_size": int(VECTOR_SIZE),
    "total_chunks_indexados": int(len(points)),
    "batch_size": int(BATCH_SIZE),
    "tiempo_carga_modelo_segundos": round(load_time, 4),
    "tiempo_generacion_embeddings_segundos": round(embedding_time, 4),
    "tiempo_indexacion_segundos": round(indexing_time, 4),
    "distancia": "COSINE",
}

report_path = REPORTS_DIR / f"reporte_indexacion_vectorial_{timestamp}.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)

print(f"Reporte guardado en: {report_path}")
print(json.dumps(report, ensure_ascii=False, indent=4))


## 17. Resultado final

Si las búsquedas devuelven fragmentos razonables, la indexación vectorial está funcionando correctamente.

Siguiente etapa:

- implementar búsqueda lexical con BM25;
- fusionar resultados con RRF;
- construir búsqueda híbrida;
- integrar el modelo generativo.


In [ ]:
print("Notebook 04 completado.")
print("La base vectorial está lista para pruebas de recuperación semántica.")
print("Siguiente paso recomendado: Notebook 05 - Búsqueda híbrida y pruebas RAG.")
